# 08c_ex8_approve_and_refresh.sql  Exercise 8-6：データ整備が経営ダッシュボードに反映されることを確認する（第2回・5分）

前提 : 08b でダッシュボードを配置し、画面を開いていること  
流れ : マスタ管理担当が TH-Z999 の根拠（Exercise 5B で 88 点）を確認して承認する  
→ ダッシュボードを再読み込みすると、売上・データ信頼度・打ち手の一覧が変わる  

In [ ]:
%run ./00_config

### 8-6-1. 承認前の数字を控える

In [ ]:
%sql
SELECT 'before' AS timing, unresolved_sales_amount, sales_amount_coverage, pending_candidates, trust_level
FROM v_data_trust_summary;

### 8-6-2. 担当者が根拠を確認して承認する（役割名で記録。根拠は alias_evidence）

In [ ]:
%sql
UPDATE vehicle_alias_master
SET approval_status = 'APPROVED',
    mapping_method  = 'HUMAN_REVIEWED',
    approved_by     = 'タイ営業企画 マスタ管理担当（架空）',
    source_document = 'alias_evidence（出荷実績・原価シート・担当者メモ・クエリ履歴）',
    remarks         = concat(coalesce(remarks, ''), ' / 根拠を確認して承認'),
    updated_at      = current_timestamp()
WHERE alias_id = 'A018';

### 8-6-3. 承認後の数字（View は自動で最新の対応を反映する。再集計の作業は不要）

In [ ]:
%sql
SELECT 'after' AS timing, unresolved_sales_amount, sales_amount_coverage, pending_candidates, trust_level
FROM v_data_trust_summary;

In [ ]:
%sql
SELECT MEASURE(`実績売上`) AS actual_sales, MEASURE(`販売台数`) AS sales_volume
FROM mv_vehicle_profitability
WHERE `共通機種ID` = 'VEHICLE-001';
-- 期待値: 未変換の売上が 150 百万円減り、VEHICLE-001 の販売台数が 60 台増える（TH-Z999 の 2025-07 分）

### 8-6-4. ダッシュボードをブラウザで再読み込みして、KPI・データ信頼度・打ち手の一覧が変わったことを確認する